Modelo: Qwen/Qwen2.5-Coder-1.5B-Instruct, ajuste real con QLoRA.

## GPU

In [ ]:
import torch
print("PyTorch:",torch.__version__)
print("CUDA disponible:",torch.cuda.is_available())
if torch.cuda.is_available():
 print("GPU:",torch.cuda.get_device_name(0))
 print("VRAM GB:",round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
assert torch.cuda.is_available(), "Activa GPU en Entorno de ejecución > Cambiar tipo de entorno"

PyTorch: 2.14.1+cu130
CUDA disponible: True
GPU: Tesla T4
VRAM GB: 14.56


## Librerías
Ejecutar una sola vez por sesión. Si Colab pide reiniciar, reiniciar y volver a comenzar desde la celda 1.

In [ ]:
%pip install -q "pandas==2.2.3" "huggingface-hub>=0.34,<1.0" "transformers" "datasets" "accelerate" "peft" "trl" "bitsandbytes" "scikit-learn"

In [ ]:
import pandas
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("Pandas:", pandas.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Pandas: 2.2.3
PyTorch: 2.14.1+cu130
Transformers: 4.57.6
Datasets: 5.0.1
PEFT: 0.21.3
TRL: 0.24.0
CUDA disponible: True
GPU: Tesla T4


## dataset.zip

In [ ]:
from google.colab import files
from pathlib import Path
if not Path("/content/dataset.zip").exists():
 uploaded=files.upload()
 assert "dataset.zip" in uploaded, "Debes subir el archivo dataset.zip"
print("ZIP listo:",Path("/content/dataset.zip").stat().st_size/1024**2,"MB")

ZIP listo: 39.273343086242676 MB


In [ ]:
import zipfile

ruta = "/content/dataset.zip"

with zipfile.ZipFile(ruta, "r") as zip_ref:
    archivos = zip_ref.namelist()

print("Total de archivos:", len(archivos))
print("\nContenido del ZIP:")

for archivo in archivos[:30]:
    print(archivo)

Total de archivos: 18

Contenido del ZIP:
dataset/
__MACOSX/._dataset
dataset/type-5.csv
__MACOSX/dataset/._type-5.csv
dataset/type-4.csv
__MACOSX/dataset/._type-4.csv
dataset/type-3.csv
__MACOSX/dataset/._type-3.csv
dataset/type-2.csv
__MACOSX/dataset/._type-2.csv
dataset/type-1.csv
__MACOSX/dataset/._type-1.csv
dataset/clone.csv
__MACOSX/dataset/._clone.csv
dataset/id2sourcecode.zip
__MACOSX/dataset/._id2sourcecode.zip
dataset/nonclone.csv
__MACOSX/dataset/._nonclone.csv


In [ ]:
import pandas as pd
import zipfile
import io

ruta = "/content/dataset.zip"

tipos = {}

with zipfile.ZipFile(ruta, "r") as z:
    for i in range(1, 5):
        nombre = f"dataset/type-{i}.csv"
        tipos[i] = pd.read_csv(
            io.BytesIO(z.read(nombre)),
            header=None,
            names=["id1", "id2", "etiqueta", "sim1", "sim2"]
        )

    no_clones = pd.read_csv(
        io.BytesIO(z.read("dataset/nonclone.csv"))
    )

for i in range(1, 5):
    print(f"Tipo {i}: {len(tipos[i])} pares")
    print("Etiquetas:", tipos[i]["etiqueta"].unique())

print("\nNo clones:", len(no_clones))
print("Columnas:", no_clones.columns.tolist())


Tipo 1: 48116 pares
Etiquetas: [1]
Tipo 2: 4234 pares
Etiquetas: [2]
Tipo 3: 21395 pares
Etiquetas: [3]
Tipo 4: 86341 pares
Etiquetas: [3]

No clones: 279032
Columnas: ['FUNCTION_ID_ONE', 'FUNCTION_ID_TWO']


## Muestra

In [ ]:
usados = set()

def seleccionar_balanceado(cantidades):
    seleccion = []

    for tipo, cantidad in cantidades.items():
        candidatos = datos[datos["tipo"] == tipo]
        candidatos = candidatos.sample(frac=1, random_state=25)

        contador = 0

        for fila in candidatos.itertuples(index=False):
            if fila.id1 in usados or fila.id2 in usados:
                continue

            seleccion.append({
                "id1": fila.id1,
                "id2": fila.id2,
                "tipo": fila.tipo,
                "label": fila.label
            })

            usados.add(fila.id1)
            usados.add(fila.id2)
            contador += 1

            if contador == cantidad:
                break

        if contador < cantidad:
            raise ValueError(f"Faltan pares del tipo {tipo}")

    return pd.DataFrame(seleccion).sample(
        frac=1, random_state=25
    ).reset_index(drop=True)

# Reservar primero los datos de prueba
test = seleccionar_balanceado({
    0: 50, 1: 13, 2: 12, 3: 12, 4: 13
})

val = seleccionar_balanceado({
    0: 10, 1: 3, 2: 2, 3: 2, 4: 3
})

train = seleccionar_balanceado({
    0: 10, 1: 3, 2: 2, 3: 2, 4: 3
})

print("Entrenamiento:", len(train))
print("Validación:", len(val))
print("Prueba:", len(test))

print("\nDistribución de entrenamiento:")
print(train["tipo"].value_counts().sort_index())

print("\nDistribución de validación:")
print(val["tipo"].value_counts().sort_index())

print("\nDistribución de prueba:")
print(test["tipo"].value_counts().sort_index())

Entrenamiento: 20
Validación: 20
Prueba: 100

Distribución de entrenamiento:
tipo
0    10
1     3
2     2
3     2
4     3
Name: count, dtype: int64

Distribución de validación:
tipo
0    10
1     3
2     2
3     2
4     3
Name: count, dtype: int64

Distribución de prueba:
tipo
0    50
1    13
2    12
3    12
4    13
Name: count, dtype: int64


## Códigos Java

In [ ]:
import zipfile
import io

# Abrir el ZIP principal
with zipfile.ZipFile("/content/dataset.zip", "r") as z:
    contenido = z.read("dataset/id2sourcecode.zip")

# Abrir el ZIP que contiene los códigos Java
zip_java = zipfile.ZipFile(io.BytesIO(contenido))

archivos_java = zip_java.namelist()

print("Total de archivos:", len(archivos_java))

print("\nPrimeros 15 archivos:")
for archivo in archivos_java[:15]:
    print(archivo)

print("\nPrimer par de entrenamiento:")
print(train.iloc[0][["id1", "id2", "tipo", "label"]])

Total de archivos: 73320

Primeros 15 archivos:
id2sourcecode/
id2sourcecode/10000061.java
id2sourcecode/10000064.java
id2sourcecode/10000767.java
id2sourcecode/10000832.java
id2sourcecode/10001179.java
id2sourcecode/10001181.java
id2sourcecode/10001182.java
id2sourcecode/1000135.java
id2sourcecode/10001403.java
id2sourcecode/10001570.java
id2sourcecode/10001843.java
id2sourcecode/100043.java
id2sourcecode/1000504.java
id2sourcecode/1000514.java

Primer par de entrenamiento:
id1      4518933
id2      6244937
tipo           2
label          1
Name: 0, dtype: int64


In [ ]:
# Función para obtener el código Java
def obtener_codigo(id):
    ruta = f"id2sourcecode/{id}.java"

    try:
        return zip_java.read(ruta).decode("utf-8")
    except KeyError:
        return None

# Agregar los códigos a cada conjunto
for df in [train, val, test]:
    df["codigo1"] = df["id1"].apply(obtener_codigo)
    df["codigo2"] = df["id2"].apply(obtener_codigo)

# Comprobar si faltan códigos
for nombre, df in [
    ("Entrenamiento", train),
    ("Validación", val),
    ("Prueba", test)
]:
    faltantes = df["codigo1"].isna().sum() + df["codigo2"].isna().sum()

    print(nombre)
    print("Pares:", len(df))
    print("Códigos faltantes:", faltantes)
    print()

# Mostrar un ejemplo
print("PRIMER CÓDIGO JAVA:")
print(train.iloc[0]["codigo1"][:1000])

print("\nSEGUNDO CÓDIGO JAVA:")
print(train.iloc[0]["codigo2"][:1000])

print("\nEtiqueta:", train.iloc[0]["label"])
print("Tipo:", train.iloc[0]["tipo"])

Entrenamiento
Pares: 20
Códigos faltantes: 0

Validación
Pares: 20
Códigos faltantes: 0

Prueba
Pares: 100
Códigos faltantes: 0

PRIMER CÓDIGO JAVA:
    public static boolean encodeFileToFile(String infile, String outfile) {
        boolean success = false;
        java.io.InputStream in = null;
        java.io.OutputStream out = null;
        try {
            in = new Base64.InputStream(new java.io.BufferedInputStream(new java.io.FileInputStream(infile)), Base64.ENCODE);
            out = new java.io.BufferedOutputStream(new java.io.FileOutputStream(outfile));
            byte[] buffer = new byte[65536];
            int read = -1;
            while ((read = in.read(buffer)) >= 0) {
                out.write(buffer, 0, read);
            }
            success = true;
        } catch (java.io.IOException exc) {
            exc.printStackTrace();
        } finally {
            try {
                in.close();
            } catch (Exception exc) {
            }
            try {
      

## Prompts

In [ ]:
# Instrucción para el modelo
instruccion = """You are an expert in Java code clone detection.
Analyze the following two Java code fragments.
Determine whether they are code clones.
Answer only YES or NO."""

def crear_prompt(codigo1, codigo2):
    return f"""{instruccion}

Code fragment 1:
```java
{codigo1}
```

Code fragment 2:
```java
{codigo2}
```"""

# Preparar los tres conjuntos
for df in [train, val, test]:
    df["prompt"] = df.apply(
        lambda fila: crear_prompt(
            fila["codigo1"],
            fila["codigo2"]
        ),
        axis=1
    )

    df["respuesta"] = df["label"].map({
        1: "YES",
        0: "NO"
    })

# Comprobar el primer ejemplo
print("PRIMER PROMPT:")
print(train.iloc[0]["prompt"][:1200])

print("\nRESPUESTA ESPERADA:")
print(train.iloc[0]["respuesta"])

# Revisar longitud de los prompts
for nombre, df in [
    ("Entrenamiento", train),
    ("Validación", val),
    ("Prueba", test)
]:
    longitudes = df["prompt"].str.len()

    print(f"\n{nombre}:")
    print("Promedio de caracteres:", round(longitudes.mean()))
    print("Máximo de caracteres:", longitudes.max())

PRIMER PROMPT:
You are an expert in Java code clone detection.
Analyze the following two Java code fragments.
Determine whether they are code clones.
Answer only YES or NO.

Code fragment 1:
```java
    public static boolean encodeFileToFile(String infile, String outfile) {
        boolean success = false;
        java.io.InputStream in = null;
        java.io.OutputStream out = null;
        try {
            in = new Base64.InputStream(new java.io.BufferedInputStream(new java.io.FileInputStream(infile)), Base64.ENCODE);
            out = new java.io.BufferedOutputStream(new java.io.FileOutputStream(outfile));
            byte[] buffer = new byte[65536];
            int read = -1;
            while ((read = in.read(buffer)) >= 0) {
                out.write(buffer, 0, read);
            }
            success = true;
        } catch (java.io.IOException exc) {
            exc.printStackTrace();
        } finally {
            try {
                in.close();
            } catch (Excep

## Cargar Modelo Qwen

In [ ]:
from transformers import AutoTokenizer

modelo_nombre = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)

print("Tokenizador cargado correctamente")

def contar_tokens(prompt):
    return len(tokenizer.encode(
        prompt,
        add_special_tokens=False
    ))

for nombre, df in [
    ("Entrenamiento", train),
    ("Validación", val),
    ("Prueba", test)
]:
    df["tokens"] = df["prompt"].apply(contar_tokens)

    print(f"\n{nombre}:")
    print("Promedio de tokens:", round(df["tokens"].mean()))
    print("Máximo de tokens:", df["tokens"].max())
    print("Prompts mayores a 2048 tokens:",
          (df["tokens"] > 2048).sum())
    print("Prompts mayores a 4096 tokens:",
          (df["tokens"] > 4096).sum())

Tokenizador cargado correctamente

Entrenamiento:
Promedio de tokens: 529
Máximo de tokens: 1440
Prompts mayores a 2048 tokens: 0
Prompts mayores a 4096 tokens: 0

Validación:
Promedio de tokens: 323
Máximo de tokens: 718
Prompts mayores a 2048 tokens: 0
Prompts mayores a 4096 tokens: 0

Prueba:
Promedio de tokens: 514
Máximo de tokens: 3167
Prompts mayores a 2048 tokens: 1
Prompts mayores a 4096 tokens: 0


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Configuración de cuantización
config_4bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Cargar el modelo
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_nombre,
    quantization_config=config_4bits,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Modelo cargado correctamente")
print("GPU:", torch.cuda.get_device_name(0))

memoria = torch.cuda.memory_allocated() / 1024**3
print("Memoria GPU utilizada:", round(memoria, 2), "GB")

Modelo cargado correctamente
GPU: Tesla T4
Memoria GPU utilizada: 2.65 GB


## Cargar QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Preparar el modelo cuantizado
modelo = prepare_model_for_kbit_training(modelo)

# Configuración de LoRA
config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

# Agregar los adaptadores LoRA
modelo = get_peft_model(modelo, config_lora)

# Mostrar los parámetros entrenables
modelo.print_trainable_parameters()

print("\nLoRA configurado correctamente")

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410

LoRA configurado correctamente


## Cargar Datos Muestra

In [ ]:
from datasets import Dataset

def preparar_datos(df):
    ejemplos = []

    for _, fila in df.iterrows():
        mensajes = [
            {
                "role": "user",
                "content": fila["prompt"]
            },
            {
                "role": "assistant",
                "content": fila["respuesta"]
            }
        ]

        texto = tokenizer.apply_chat_template(
            mensajes,
            tokenize=False
        )

        ejemplos.append({"text": texto})

    return Dataset.from_list(ejemplos)

dataset_train = preparar_datos(train)
dataset_val = preparar_datos(val)

print("Entrenamiento:", len(dataset_train))
print("Validación:", len(dataset_val))

print("\nEjemplo de entrenamiento:")
print(dataset_train[0]["text"][:500])

print("\nFinal del ejemplo:")
print(dataset_train[0]["text"][-200:])

Entrenamiento: 20
Validación: 20

Ejemplo de entrenamiento:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You are an expert in Java code clone detection.
Analyze the following two Java code fragments.
Determine whether they are code clones.
Answer only YES or NO.

Code fragment 1:
```java
    public static boolean encodeFileToFile(String infile, String outfile) {
        boolean success = false;
        java.io.InputStream in = null;
        java.io.OutputStream out = null;
        try 

Final del ejemplo:
        }
            try {
                out.close();
            } catch (Exception exc) {
            }
        }
        return success;
    }

```<|im_end|>
<|im_start|>assistant
YES<|im_end|>



## Entrenamiento

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer

# Configuración del entrenamiento
config_train = SFTConfig(
    output_dir="/content/qwen_almatrafi_e1",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_length=2048,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=25,
    completion_only_loss=True
)

# Separar instrucción y respuesta para calcular la pérdida solamente sobre YES o NO
def preparar_completions(df):
    ejemplos = []

    for _, fila in df.iterrows():
        mensajes = [
            {"role": "user", "content": fila["prompt"]}
        ]

        prompt = tokenizer.apply_chat_template(
            mensajes,
            tokenize=False,
            add_generation_prompt=True
        )

        completion = fila["respuesta"] + tokenizer.eos_token

        ejemplos.append({
            "prompt": prompt,
            "completion": completion
        })

    return Dataset.from_list(ejemplos)

dataset_train_sft = preparar_completions(train)
dataset_val_sft = preparar_completions(val)

trainer = SFTTrainer(
    model=modelo,
    args=config_train,
    train_dataset=dataset_train_sft,
    eval_dataset=dataset_val_sft,
    processing_class=tokenizer
)

print("Entrenador configurado correctamente")
print("Ejemplos entrenamiento:", len(dataset_train_sft))
print("Ejemplos validación:", len(dataset_val_sft))

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Entrenador configurado correctamente
Ejemplos entrenamiento: 20
Ejemplos validación: 20


In [ ]:
# Entrenar Qwen con 20 ejemplos
resultado = trainer.train()

print("\nEntrenamiento finalizado correctamente")
print("Pérdida final:", resultado.training_loss)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.383100,0.252642,0.861398,11198.000000,0.875000
2,0.228600,0.176456,0.857299,22396.000000,0.950000
3,0.122100,0.156104,0.854826,33594.000000,0.975000



Entrenamiento finalizado correctamente
Pérdida final: 0.30968284159898757


In [ ]:
# Guardar los adaptadores entrenados
ruta_modelo = "/content/almatrafi_e1_lora"

trainer.save_model(ruta_modelo)
tokenizer.save_pretrained(ruta_modelo)

print("Modelo E1 guardado correctamente")
print("Ruta:", ruta_modelo)

Modelo E1 guardado correctamente
Ruta: /content/almatrafi_e1_lora


## Evaluar 100 Pares Muestra

In [ ]:
import torch
from tqdm.auto import tqdm

modelo.eval()

predicciones = []
respuestas_generadas = []

for _, fila in tqdm(test.iterrows(), total=len(test)):
    mensajes = [
        {"role": "user", "content": fila["prompt"]}
    ]

    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        salida = modelo.generate(
            **entradas,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    nuevos_tokens = salida[0][entradas["input_ids"].shape[1]:]

    respuesta = tokenizer.decode(
        nuevos_tokens,
        skip_special_tokens=True
    ).strip().upper()

    respuestas_generadas.append(respuesta)

    if respuesta.startswith("YES"):
        predicciones.append(1)
    elif respuesta.startswith("NO"):
        predicciones.append(0)
    else:
        predicciones.append(-1)

test["prediccion"] = predicciones
test["respuesta_generada"] = respuestas_generadas

print("Evaluación terminada")
print("Total:", len(test))
print("Respuestas no válidas:", (test["prediccion"] == -1).sum())

print("\nPrimeras predicciones:")
print(test[["tipo", "label", "prediccion", "respuesta_generada"]].head(10))

  0%|          | 0/100 [00:00<?, ?it/s]

Evaluación terminada
Total: 100
Respuestas no válidas: 0

Primeras predicciones:
   tipo  label  prediccion respuesta_generada
0     0      0           0                 NO
1     0      0           0                 NO
2     0      0           0                 NO
3     2      1           1                YES
4     2      1           1                YES
5     0      0           0                 NO
6     1      1           1                YES
7     4      1           0                 NO
8     4      1           0                 NO
9     4      1           0                 NO


## Métricas Muestra

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_real = test["label"]
y_pred = test["prediccion"]

print("=== RESULTADOS GENERALES E1 ===")

print("Accuracy:", round(accuracy_score(y_real, y_pred), 4))
print("Precision:", round(precision_score(y_real, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_real, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_real, y_pred, zero_division=0), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_real, y_pred, labels=[0, 1]))

print("\nReporte de clasificación:")
print(classification_report(
    y_real,
    y_pred,
    labels=[0, 1],
    target_names=["No clon", "Clon"],
    zero_division=0
))

print("\n=== RESULTADOS POR TIPO DE CLON ===")

for tipo in range(1, 5):
    datos_tipo = test[test["tipo"] == tipo]

    correctos = (datos_tipo["label"] == datos_tipo["prediccion"]).sum()
    total = len(datos_tipo)

    print(f"Tipo {tipo}: {correctos}/{total} detectados")
    print(f"Recall: {correctos / total:.4f}")
    print()

=== RESULTADOS GENERALES E1 ===
Accuracy: 0.9
Precision: 0.9762
Recall: 0.82
F1-score: 0.8913

Matriz de confusión:
[[49  1]
 [ 9 41]]

Reporte de clasificación:
              precision    recall  f1-score   support

     No clon       0.84      0.98      0.91        50
        Clon       0.98      0.82      0.89        50

    accuracy                           0.90       100
   macro avg       0.91      0.90      0.90       100
weighted avg       0.91      0.90      0.90       100


=== RESULTADOS POR TIPO DE CLON ===
Tipo 1: 13/13 detectados
Recall: 1.0000

Tipo 2: 12/12 detectados
Recall: 1.0000

Tipo 3: 10/12 detectados
Recall: 0.8333

Tipo 4: 6/13 detectados
Recall: 0.4615



## Guardar Resultados Muestra

In [ ]:
import json
import os

carpeta = "/content/resultados_almatrafi"
os.makedirs(carpeta, exist_ok=True)

metricas_e1 = {
    "experimento": "E1",
    "modelo": "Qwen2.5-Coder-1.5B-Instruct",
    "entrenamiento": 20,
    "validacion": 20,
    "prueba": 100,
    "epocas": 3,
    "accuracy": 0.90,
    "precision": 0.9762,
    "recall": 0.82,
    "f1_score": 0.8913,
    "recall_por_tipo": {
        "tipo_1": 1.0,
        "tipo_2": 1.0,
        "tipo_3": 0.8333,
        "tipo_4": 0.4615
    }
}

with open(f"{carpeta}/metricas_e1.json", "w") as f:
    json.dump(metricas_e1, f, indent=4)

test[[
    "id1", "id2", "tipo", "label",
    "prediccion", "respuesta_generada"
]].to_csv(
    f"{carpeta}/predicciones_e1.csv",
    index=False
)

print("Resultados E1 guardados correctamente")
print(os.listdir(carpeta))

Resultados E1 guardados correctamente
['metricas_e1.json', 'predicciones_e1.csv']


In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# Carpeta dentro de Mi unidad
carpeta = "/content/drive/MyDrive/Almatrafi/Experimento_E1"
os.makedirs(carpeta, exist_ok=True)

# Guardar resultados
for archivo in ["metricas_e1.json", "predicciones_e1.csv"]:
    shutil.copy(
        f"/content/resultados_almatrafi/{archivo}",
        carpeta
    )

# Guardar conjuntos utilizados
for nombre, df in [("train", train), ("val", val), ("test", test)]:
    df[["id1", "id2", "tipo", "label"]].to_csv(
        f"{carpeta}/{nombre}_e1.csv",
        index=False
    )

# Guardar modelo entrenado
shutil.copytree(
    "/content/almatrafi_e1_lora",
    f"{carpeta}/modelo_lora",
    dirs_exist_ok=True
)

print("Guardado correctamente en Mi unidad")
print("Carpeta:", carpeta)
print("Archivos:", os.listdir(carpeta))

Mounted at /content/drive
Guardado correctamente en Mi unidad
Carpeta: /content/drive/MyDrive/Almatrafi/Experimento_E1
Archivos: ['metricas_e1.json', 'predicciones_e1.csv', 'train_e1.csv', 'val_e1.csv', 'test_e1.csv', 'modelo_lora']
